# مسار الفونيمات للأسماء المُعْرَبة فقط (بدون أفعال)

هذا الدفتر يبني مسارًا تعليميًا لاستخراج الدالّ الصوتي (حرف + حركة مُلصقة للحرف السابق Affix-to-Base)، ثم تقطيع المقاطع ضمن بوابات 6 (CV/CVC/CVV/CVVC/CVCC/VC)، مع تصفية الأصناف المبنية (ضمائر/أدوات/أسماء الإشارة/الموصول/الاستفهام) واستبعاد الأفعال، لاستخراج الأسماء المُعربة فقط. بعدها نحسب حركة الإعراب النهائية ومقاطع النهاية وإحصاءات إجمالية.

In [2]:
# -*- coding: utf-8 -*-
# 1) استيراد المكتبات والتهيئة (CONFIG)

from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict, Any
import re, json
import numpy as np
import pandas as pd

# حدّث المسار حسب بيئتك
TEXT_PATH = "./data/quran.txt"  # غيّره لمسارك، أو اتركه ليستخدم عينة بديلة
OUT_PREFIX = "./reports/noun_muarab/noun_muarab_path"

# إنشاء مجلد الإخراج إن لزم
Path(OUT_PREFIX).parent.mkdir(parents=True, exist_ok=True)

# ========= Arabic basics =========
DIAC2LBL = {"َ":"F", "ِ":"K", "ُ":"D", "ْ":"S"}  # F=fatha, K=kasra, D=damma, S=sukun
AR_SET    = set("ابتثجحخدذرزسشصضطظعغفقكلمنهويء")
LONG_VOW  = set("اوي")
RE_DIAC   = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
RE_TATW   = re.compile(r"\u0640")


def strip_diac(s: str) -> str:
    return RE_DIAC.sub("", s)


def normalize_text(s: str, keep_diac: bool = True) -> str:
    s = RE_TATW.sub("", s)
    if not keep_diac:
        s = RE_DIAC.sub("", s)
    return s

In [3]:
# 2) تحميل النص وتطبيعه (UTF-8)

def load_text(path: str) -> str:
    p = Path(path)
    if p.exists():
        return p.read_text(encoding="utf-8", errors="ignore")
    # Fallback sample (آيتان مع تشكيل لأغراض العرض)
    sample = (
        "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ\n"
        "الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ\n"
        "الرَّحْمَٰنِ الرَّحِيمِ\n"
        "مَالِكِ يَوْمِ الدِّينِ\n"
        "إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ\n"
    )
    return sample


text_raw = load_text(TEXT_PATH)
text = normalize_text(text_raw, keep_diac=True)
words = [w for w in re.split(r"\s+", text) if w.strip()]
len(words), words[:10]

(17,
 ['بِسْمِ',
  'اللَّهِ',
  'الرَّحْمَٰنِ',
  'الرَّحِيمِ',
  'الْحَمْدُ',
  'لِلَّهِ',
  'رَبِّ',
  'الْعَالَمِينَ',
  'الرَّحْمَٰنِ',
  'الرَّحِيمِ'])

In [4]:
# 3) تفريغ الدال الصوتي: Affix-to-Base

from dataclasses import dataclass
from typing import Optional, List


def ar_tokens(s: str) -> List[str]:
    # نقتصر على الحروف العربية، الحركات، والمسافات
    return [ch for ch in s if (ch in AR_SET or ch in DIAC2LBL or ch.isspace())]


@dataclass
class CharDia:
    ch: str
    dia: Optional[str]
    raw: str
    idx: int


def affix_to_base(tokens: List[str]) -> List[CharDia]:
    out: List[CharDia] = []
    last: Optional[int] = None
    for i, t in enumerate(tokens):
        if t in AR_SET:
            out.append(CharDia(ch=t, dia=None, raw=t, idx=i))
            last = len(out) - 1
        elif t in DIAC2LBL:
            if last is not None:
                out[last].dia = DIAC2LBL[t]
                out[last].raw += t
        else:
            last = None
    return out


# اختبار سريع على كلمة مُشكّلة
_test = "الْعالَمِينَ"
_test_chars = affix_to_base(ar_tokens(_test))
[(c.ch, c.dia) for c in _test_chars]

[('ا', None),
 ('ل', 'S'),
 ('ع', None),
 ('ا', None),
 ('ل', 'F'),
 ('م', 'K'),
 ('ي', None),
 ('ن', 'F')]

In [5]:
# 4) بناء المقاطع: بوابات 6 (CV/CVC/CVV/CVVC/CVCC/VC)

from dataclasses import dataclass
from typing import List

SYLL_GATES = ["CV", "CVC", "CVV", "CVVC", "CVCC", "VC"]


@dataclass
class Syllable:
    text: str
    gate: str
    state: str
    start: int
    end: int


def syllabify(chars: List[CharDia]) -> List[Syllable]:
    def is_c(j: int) -> bool:
        return 0 <= j < len(chars) and (chars[j].ch in AR_SET)

    def has_v(j: int) -> bool:
        return 0 <= j < len(chars) and (chars[j].dia in ("F", "K", "D"))

    def is_long(j: int) -> bool:
        return 0 <= j < len(chars) and (chars[j].ch in LONG_VOW)

    def raw(a: int, b: int) -> str:
        return "".join(c.raw for c in chars[a : b + 1])

    out: List[Syllable] = []
    i = 0
    while i < len(chars):
        if is_c(i) and has_v(i):
            start = i
            state = chars[i].dia  # F/K/D
            j = i + 1

            # CVV ± closers
            if is_long(j):
                end = j
                gate = "CVV"
                j += 1
                if is_c(j):
                    gate = "CVVC"
                    end = j
                    j += 1
                if is_c(j):
                    gate = "CVCC"
                    end = j
                    j += 1
                out.append(Syllable(raw(start, end), gate, state, start, end))
                i = j
                continue

            # CVC / CVCC
            if is_c(j):
                end = j
                gate = "CVC"
                j += 1
                if is_c(j):
                    gate = "CVCC"
                    end = j
                    j += 1
                out.append(Syllable(raw(start, end), gate, state, start, end))
                i = j
                continue

            # CV
            out.append(Syllable(raw(start, i), "CV", state, start, i))
            i = i + 1
            continue
        i += 1

    return [s for s in out if s.gate in SYLL_GATES]


# تحقق سريع
_test_sylls = syllabify(_test_chars)
[(s.text, s.gate, s.state) for s in _test_sylls]

[('لَمِي', 'CVCC', 'F'), ('نَ', 'CV', 'F')]

In [6]:
# 5) مرشحات المبني واستبعاد الأفعال

# Pronouns (built): detached/attached
DET_PRON = set(["انا","نحن","انت","انتي","انتما","انتم","انتن","هو","هي","هما","هم","هن"])
ATT_PRON = ["كما","كم","كن","نا","ني","ه","ها","هم","هن","ي"]


def norm_core(s: str) -> str:
    return (
        s.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ٱ", "ا")
        .replace("ى", "ي")
        .replace("\u0640", "")
    )


def is_detached_pronoun(word: str) -> bool:
    return norm_core(strip_diac(word)) in DET_PRON


def match_attached_pronoun(word: str) -> Optional[str]:
    core = norm_core(strip_diac(word))
    for suf in sorted(ATT_PRON, key=len, reverse=True):
        if core.endswith(suf) and len(core) > len(suf):
            return suf
    return None


# Particles (built) — نبقيها بسيطة؛ سنستبعدها
PARTICLES = set(["و","ف","ثم","بل","لكن","او","ام","على","من","الى","عن","في","ب","ك","ل","حتى","كي","لان","اذن"])


def is_particle(word: str) -> bool:
    return norm_core(strip_diac(word)) in PARTICLES


# Verb heuristics (استبعاد)
VERB_PRESENT_PREFIX = ["ي", "ت", "ا", "ن"]
VERB_PAST_SUFFIX_NOD = [strip_diac(x) for x in ["تُ", "تَ", "تِ", "تْ", "نا", "تما", "تم", "تن", "ا", "وا", "نَ"]]


def looks_like_verb(word: str) -> bool:
    base = norm_core(word)
    nod = strip_diac(base)
    if any(base.startswith(p) or nod.startswith(p) for p in VERB_PRESENT_PREFIX):
        return True
    if any(nod.endswith(s) for s in VERB_PAST_SUFFIX_NOD):
        return True
    return False


# Demonstratives / Relatives / Interrogatives (built) — استبعاد سريع
DEMONS = set(["هذا", "هذه", "هذان", "هاتان", "هؤلاء", "ذلك", "تلك", "اولئك"])
RELAT = set(["الذي", "التي", "الذين", "اللاتي", "اللائي", "من", "ما"])
INTER = set(["من", "ما", "أين", "متى", "كيف", "كم", "أي"])


def is_built_noun_other(word: str) -> bool:
    w = norm_core(strip_diac(word))
    return (w in DEMONS) or (w in RELAT) or (w in INTER)


# اختبارات سريعة (تعليقات):
# assert is_particle("في")
# assert is_detached_pronoun("هو")
# assert match_attached_pronoun("كتابُهُ") == "ه"

In [7]:
# 6) مصنّف الاسم المُعرب

def is_noun_muarab(word: str) -> bool:
    # استبعد المبني (ضمائر/أدوات/أسماء مبنية)، واستبعد الأفعال
    if is_detached_pronoun(word) or match_attached_pronoun(word):
        return False
    if is_particle(word) or is_built_noun_other(word):
        return False
    if looks_like_verb(word):
        return False
    # المتبقّي: مرشح اسم مُعْرَب (قد يشمل الممنوع من الصرف؛ ما يزال معرباً من حيث الأصل)
    return True


# اختبارات سريعة توثيقية
cands = [
    "كتابٌ",  # اسم معرب
    "هذا",    # اسم إشارة (مبني)
    "في",     # أداة (حرف جر)
    "يكتبُ",  # فعل مضارع
]
[(w, is_noun_muarab(w)) for w in cands]

[('كتابٌ', True), ('هذا', False), ('في', False), ('يكتبُ', False)]

In [8]:
# 7) خط الأنابيب والإحصاءات

def analyze_noun_muarab(text_path: str) -> Dict[str, Any]:
    text_raw = Path(text_path).read_text(encoding="utf-8", errors="ignore") if Path(text_path).exists() else text
    doc = normalize_text(text_raw, keep_diac=True)
    words_local = [w for w in re.split(r"\s+", doc) if w.strip()]

    rows: List[Dict[str, Any]] = []
    diac_counts_global: Dict[str, int] = {"F": 0, "K": 0, "D": 0, "S": 0}
    gate_counts_global: Dict[str, int] = {g: 0 for g in SYLL_GATES}

    for w in words_local:
        if not is_noun_muarab(w):
            continue
        # phoneme+diac for the word
        chars = affix_to_base(ar_tokens(w))
        sylls = syllabify(chars)
        # diacritics list (token-level)
        dseq = [c.dia for c in chars if c.dia]
        for d in dseq:
            diac_counts_global[d] = diac_counts_global.get(d, 0) + 1
        for s in sylls:
            gate_counts_global[s.gate] = gate_counts_global.get(s.gate, 0) + 1

        # final (case-like) diacritic heuristic: آخر حركة ظاهرة في الكلمة
        final_diac = None
        for c in reversed(chars):
            if c.dia in ("F", "K", "D", "S"):
                final_diac = c.dia
                break

        rows.append(
            {
                "word": w,
                "len": len(w),
                "diacritics": dseq,
                "syllables": [s.gate for s in sylls],
                "final_diac": final_diac,  # تقريب: F=نصب، K=جر، D=رفع، S=وقف/سكون
            }
        )

    df = pd.DataFrame(rows)
    noun_count = len(df)

    # نسب موجزة
    def pct(x: int, tot: int) -> float:
        return round(100.0 * (x / (tot or 1)), 3)

    summary: Dict[str, Any] = {
        "total_tokens": len(words_local),
        "noun_muarab_tokens": noun_count,
        "noun_muarab_%": pct(noun_count, len(words_local)),
        "final_diac_%": df["final_diac"].value_counts(normalize=True, dropna=True).to_dict() if noun_count else {},
        "diacritics_counts": diac_counts_global,
        "syllable_gates_counts": gate_counts_global,
    }
    return {"df": df, "summary": summary}


# تشغيل سريع على النص المحمّل/العينة
_res = analyze_noun_muarab(TEXT_PATH)
_df = _res["df"]
_summary = _res["summary"]

print("Total tokens:", _summary["total_tokens"])  # إجمالي الكلمات بعد التقسيم
print("Mu'rab noun tokens:", _summary["noun_muarab_tokens"], f"({_summary['noun_muarab_%']}%)")
print("Final-diac % (≈ case endings):", _summary["final_diac_%"])

Total tokens: 17
Mu'rab noun tokens: 4 (23.529%)
Final-diac % (≈ case endings): {'K': 0.75, 'F': 0.25}


In [9]:
# 8) التنفيذ والعرض في VS Code Notebook

# عرض ملخصات إضافية ومعاينة الجدول
print("Diacritics counts (F/K/D/S) total:", _summary["diacritics_counts"])
print("Syllable gates counts:", _summary["syllable_gates_counts"])

# عرض أول صفوف الجدول للتحقق من الأعمدة
_df.head(10)

Diacritics counts (F/K/D/S) total: {'F': 4, 'K': 6, 'D': 0, 'S': 1}
Syllable gates counts: {'CV': 0, 'CVC': 1, 'CVV': 0, 'CVVC': 0, 'CVCC': 3, 'VC': 0}


,word,len,diacritics,syllables,final_diac
0,بِسْمِ,6,"[K, S, K]",[CVCC],K
1,رَبِّ,5,"[F, K]",[CVC],K
2,مَالِكِ,7,"[F, K, K]",[CVCC],K
3,وَإِيَّاكَ,10,"[K, F, F]",[CVCC],F


In [10]:
# 9) الحفظ إلى CSV/JSON

rows_path = f"{OUT_PREFIX}_rows.csv"
sum_path = f"{OUT_PREFIX}_summary.json"

# حفظ ملفات للتوثيق/التحليل
Path(rows_path).write_text(_df.to_csv(index=False), encoding="utf-8")
Path(sum_path).write_text(json.dumps(_summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\nSaved files:\n- {rows_path}\n- {sum_path}")


Saved files:
- ./reports/noun_muarab/noun_muarab_path_rows.csv
- ./reports/noun_muarab/noun_muarab_path_summary.json


In [11]:
# 10) رسوم بيانية اختيارية لتوزيع الحركات والمقاطع

import matplotlib.pyplot as plt
import seaborn as sns

# تحويل القواميس إلى Series/Frame
final_diac_series = pd.Series(_summary.get("final_diac_%", {}))
syll_counts = pd.Series(_summary.get("syllable_gates_counts", {}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# رسم final_diac
if not final_diac_series.empty:
    final_diac_series.sort_index().plot(kind="bar", ax=axes[0], rot=0, color="#7aa6c2")
    axes[0].set_title("توزيع الحركة النهائية للأسماء المُعربة")
    axes[0].set_xlabel("الحركة النهائية (F/K/D/S)")
    axes[0].set_ylabel("النسبة")
else:
    axes[0].text(0.5, 0.5, "لا توجد بيانات", ha="center")
    axes[0].set_axis_off()

# رسم المقاطع
if not syll_counts.empty:
    syll_counts = syll_counts.reindex(["CV", "CVC", "CVV", "CVVC", "CVCC", "VC"]).fillna(0)
    (syll_counts / syll_counts.sum()).plot(kind="bar", ax=axes[1], rot=0, color="#c27a9b")
    axes[1].set_title("توزيع بوابات المقاطع للأسماء المُعربة")
    axes[1].set_xlabel("البوابات")
    axes[1].set_ylabel("النسبة")
else:
    axes[1].text(0.5, 0.5, "لا توجد بيانات", ha="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'